# Common Setup and Functions

## Imports

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import duckdb

## Paths

In [ ]:
@dataclass
class ProjectPaths:
    raw_dir: Path
    db_path: Path
    output_dir: Path


ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "notebooks" / "00_common.ipynb").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook inside the Olist project folder.")

paths = ProjectPaths(
    raw_dir=ROOT / "data" / "raw",
    db_path=ROOT / "database" / "olist.duckdb",
    output_dir=ROOT / "outputs",
)

## Source Files

In [ ]:
REQUIRED_SOURCE_FILES = (
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
)


def validate_sources(paths):
    missing = [name for name in REQUIRED_SOURCE_FILES if not (paths.raw_dir / name).is_file()]
    if missing:
        raise FileNotFoundError("Missing source files: " + ", ".join(missing))

## SQL Phases

In [ ]:
SQL_PHASES = {
    "load": ("00_staging",),
    "build": ("10_intermediate", "20_marts"),
    "validate": ("30_data_checks",),
    "analyze": ("40_analysis",),
}

sql_notebook = str(ROOT / "notebooks" / "00_sql_definitions.ipynb")
%run "$sql_notebook"

## Database Access

In [ ]:
def _sql_literal(value):
    return str(value).replace("'", "''")


def connect(paths, read_only=False):
    if not read_only:
        paths.db_path.parent.mkdir(parents=True, exist_ok=True)
    return duckdb.connect(str(paths.db_path), read_only=read_only)


def query(sql):
    with connect(paths, read_only=True) as connection:
        return connection.execute(sql).df()

## Run a Phase

In [ ]:
def run_phase(paths, phase):
    if phase not in SQL_PHASES:
        raise ValueError(f"Unknown phase: {phase}")
    if phase == "load":
        validate_sources(paths)

    with connect(paths) as con:
        con.execute("PRAGMA threads=4")
        # Keep each SQL phase complete if a query fails.
        con.execute("BEGIN TRANSACTION")
        try:
            for name in SQL_PHASES[phase]:
                sql = SQL_SCRIPTS[name].replace("{{RAW_DIR}}", _sql_literal(paths.raw_dir))
                con.execute(sql)
                print(f"[{phase}] {name}")
            con.execute("COMMIT")
        except Exception:
            con.execute("ROLLBACK")
            raise

## Check Data Quality

In [ ]:
def check_data(paths):
    with connect(paths, read_only=True) as con:
        try:
            results = con.execute(
                "SELECT check_id, affected_count FROM checks.required_checks ORDER BY check_id"
            ).fetchall()
        except duckdb.Error as exc:
            raise RuntimeError("Run data loading, preprocessing, and data checks first.") from exc

    expected = {f"B{number:03d}" for number in range(1, 12)}
    if len(results) != 11 or {row[0] for row in results} != expected:
        raise RuntimeError("The 11 required data checks are missing or incomplete.")
    failed = [check_id for check_id, count in results if count is None or count != 0]
    if failed:
        raise RuntimeError("Data checks failed: " + ", ".join(failed))

## Export Aggregates

In [ ]:
def export_tableau(paths, tables):
    check_data(paths)
    tableau_dir = paths.output_dir / "tableau"
    tableau_dir.mkdir(parents=True, exist_ok=True)
    with connect(paths, read_only=True) as con:
        for name, (table, order_by) in tables.items():
            target = tableau_dir / f"{name}.csv"
            columns = con.execute(f"DESCRIBE SELECT * FROM {table}").fetchall()
            selections = []
            for column, data_type, *_ in columns:
                quoted = '"' + column.replace('"', '""') + '"'
                # Round floating-point values only in the exported CSV.
                if data_type in {"FLOAT", "REAL", "DOUBLE"}:
                    selections.append(f"ROUND({quoted}, 10) AS {quoted}")
                else:
                    selections.append(quoted)
            projection = ", ".join(selections)
            con.execute(
                f"COPY (SELECT {projection} FROM {table} ORDER BY {order_by}) "
                f"TO '{_sql_literal(target)}' (HEADER, DELIMITER ',')"
            )
            print(f"[export] {target.name}")